> 在线讲义：[第 19 章 · LayerNorm：把每个词的数字调到统一范围](https://wikieden.github.io/transformer-visual/lessons/19-layernorm.html) · 本书《从零读懂 Transformer》

# 第 19 章 · LayerNorm：把每个词的数字调到统一范围

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/19-layernorm.html) · 仓库里 `transformer-deep-dive/notes/lessons/19-layernorm.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 平均值 | $\mu$ | `x.mean(-1)` | [2, 4, 6, 8] → 5 |
| 方差 | $\sigma^2$，除以 d | `x.var(-1, correction=0)` | 5；[80, 100] → 100 |
| 整理 | $(x-\mu)/\sqrt{\sigma^2+\varepsilon}$ | 手写 / `F.layer_norm` | [−1.342, −0.447, 0.447, 1.342] |
| ε 兜底 | [7, 7] | 同上 | [0, 0]，没有 nan |
| γ、β | $\gamma\hat x + \beta$ | `gamma * x_hat + beta` | [−1, −3, 1.5, 3.5] |
| 初始值 | γ 全 1、β 全 0 | `nn.LayerNorm(4).weight / .bias` | [1, 1, 1, 1] / [0, 0, 0, 0] |
| RMSNorm | $x / \text{RMS}(x)$ | 手写 / `nn.RMSNorm` | [1, 7] → [0.2, 1.4] |
| pre-LN 主干不整理 | $x + \text{Sublayer}(\text{LN}(x))$ | `pre_ln(...) − x` | 正好等于批注 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
EPS = 1e-5

**函数 `layer_norm_by_hand`**：γ · (x − μ) / √(σ² + ε) + β，μ、σ² 只用这个词自己的数（最后一维）。

In [ ]:
def layer_norm_by_hand(x: torch.Tensor, gamma=None, beta=None, eps: float = EPS) -> torch.Tensor:
    """γ · (x − μ) / √(σ² + ε) + β，μ、σ² 只用这个词自己的数（最后一维）。"""
    mu = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, correction=0)  # 除以 d，不是 d − 1
    x_hat = (x - mu) / torch.sqrt(var + eps)
    if gamma is not None:
        x_hat = gamma * x_hat + beta
    return x_hat

**函数 `rms_norm_by_hand`**：x / √(mean(x²) + ε) · g：不减平均值，也没有 β。

In [ ]:
def rms_norm_by_hand(x: torch.Tensor, g=None, eps: float = EPS) -> torch.Tensor:
    """x / √(mean(x²) + ε) · g：不减平均值，也没有 β。"""
    rms = torch.sqrt((x**2).mean(dim=-1, keepdim=True) + eps)
    return x / rms * (1 if g is None else g)

**函数 `post_ln`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def post_ln(x, sublayer, ln):
    return ln(x + sublayer(x))

**函数 `pre_ln`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def pre_ln(x, sublayer, ln):
    return x + sublayer(ln(x))

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 3) -> list[float]:
    return [round(float(v), n) + 0.0 for v in t.detach().flatten()]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
a, b = torch.tensor([80.0, 90, 100]), torch.tensor([50.0, 60, 70])
print("减平均值:", r(a - a.mean()), r(b - b.mean()))
x = torch.tensor([80.0, 100.0])
print(
    "\n方差 [80, 100]：torch.var 默认除以 d − 1 =",
    float(x.var()),
    "；LayerNorm 要的是除以 d =",
    float(x.var(correction=0)),
)
w = torch.tensor([2.0, 4.0, 6.0, 8.0])

In [ ]:
print("\n[2, 4, 6, 8]: μ =", float(w.mean()), " 减完", r(w - w.mean()), " σ² =", float(w.var(correction=0)))
print("  手写  :", r(layer_norm_by_hand(w)))
print("  F.layer_norm:", r(F.layer_norm(w, (4,), eps=EPS)))
print("[7, 7] 有 ε 兜底:", r(layer_norm_by_hand(torch.tensor([7.0, 7.0]))), "（没有 nan）")
x_hat = torch.tensor([-1.0, -1, 1, 1])
gamma, beta = torch.tensor([1.0, 3, 1, 3]), torch.tensor([0.0, 0, 0.5, 0.5])

In [ ]:
print("\nγ x̂ + β =", r(gamma * x_hat + beta))
ln = nn.LayerNorm(4)
print("nn.LayerNorm(4) 一开始: γ =", ln.weight.tolist(), " β =", ln.bias.tolist(), "（先不改动）")
sent = torch.tensor([[2.0, 4, 6, 8], [80, 90, 100, 110], [1, 1, 1, 5]])
out = ln(sent)
print("3 个词一起进：每个词各自整理，每行平均", r(out.mean(-1)), " 方差", r(out.var(-1, correction=0), 2))
print(
    "\nRMSNorm [1, 7]:",
    r(rms_norm_by_hand(torch.tensor([1.0, 7.0])), 2),
    "  nn.RMSNorm:",
    r(nn.RMSNorm(2, eps=EPS)(torch.tensor([1.0, 7.0])), 2),
)
print("LayerNorm [1, 7]:", r(layer_norm_by_hand(torch.tensor([1.0, 7.0])), 2))

In [ ]:
print("\npost-LN vs pre-LN（子层 = 一个线性层）")
torch.manual_seed(0)
sub, ln = nn.Linear(4, 4), nn.LayerNorm(4)
x = torch.randn(4) * 10
with torch.no_grad():
    print("  post-LN 输出被整理过，平均 ≈", round(float(post_ln(x, sub, ln).mean()), 3))
    print(
        "  pre-LN 输出 − 输入 == 批注 Sublayer(LN(x)):",
        torch.allclose(pre_ln(x, sub, ln) - x, sub(ln(x)), atol=1e-5),
        "（主干没被整理）",
    )

## 3. 代码里才看得到的

1. **最大的坑：`torch.var` 默认除以 d − 1**：[80, 100] 默认算出 200，LayerNorm 要的是除以 d 的 100。统计学里"估计总体方差"习惯除以 d − 1，LayerNorm 不是在估计什么，所以要写 `correction=0`。
2. **`keepdim=True`**：平均值保持成"每行一个数"的竖条（n × 1），才能用广播从每一行里减掉自己的平均值。
3. **`dim=-1` 就是"每个词自己整理自己"**：3 个词一起进去，每一行各算各的 μ、σ，出来每行平均 0、方差 1。

## 4. 自己改一改

- 把 `correction=0` 删掉再跑手写版，和 `F.layer_norm` 还一样吗？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先减平均对中心，再除标准差统一宽窄；
> 分母加个 ε，防止除以 0；
> 整理完乘 γ 加 β，模型自己定大小；
> pre-LN 先整理给子层看，批注加回原件；RMSNorm 不减平均，只除 RMS。